# HydraY NNUE: small net (128 -> 8 -> 1) on an A100

Runtime -> **GPU A100** (High-RAM if offered), then "Run all".

### What it trains
`(768x4kb_hm -> 128)x2 -> pairwise -> 128 -> 8 -> 1`, the big net's architecture at an
eighth of the width: the same 4 mirrored king buckets, 8 output buckets, quantisation
and centipawn scale. Trainer: `nnue/trainer/src/bin/trainer_small.rs` on the
`small-net` branch; the engine reads the result through `SmallEvalFile`.
File size: **795,456 bytes**.

### Data
The small net only evaluates unbalanced positions, so it is trained on those alone:
the cell "filter" streams every data part through `datafilter` and keeps the records
whose material imbalance (engine `PIECE_VALUES`: P 100, N 344, B 359, R 502, Q 960) is
at least `MIN_MATERIAL`. **`MIN_MATERIAL` must not exceed the threshold of the engine's
small-net selector** (Andrea's code): a net never shown the positions just above that
threshold evaluates them blind. Measured on v8 data: >=300 keeps 20.8% of the records,
>=400 14.1%, >=500 9.3%, >=1000 1.5%.

The filtered set is cached on Drive (zstd), so a restart skips the filter.

### Steps
1. setup and configuration (edit the cell "configuration");
2. Rust, clone of `small-net`, architecture check;
3. filter (or reuse of the cached set);
4. training in stages of `STAGE` superbatches, each checkpoint copied to Drive;
5. final check and copy of the net to Drive.

If the session dies, run the cells up to "stage helpers" and then the cell "RESTART".

In [ ]:
# --- helper: any failed command stops the notebook, output streams live ---
import subprocess, os, re, glob, json

def sh(cmd):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'FAILED (exit {p.returncode}): {cmd}')

gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader',
                     shell=True, capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'none')
assert gpu, 'no GPU: change the runtime type'
if 'A100' not in gpu:
    print('WARNING: not an A100. It still trains, only slower.')
sh('nproc; free -g | head -2; df -h /content | tail -1')

In [ ]:
# --- configuration ---
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'

# Data parts on Drive (bulletformat, zstd). The v9 pool by default; the expected
# record counts catch a truncated upload or a partial decompression, which
# otherwise end without an error.
DATA_TAG = 'v9'
EXPECTED_RECORDS = {
    'hydray_v9_part1.bin.zst': 647_094_461,
    'hydray_v9_part2.bin.zst': 647_094_461,
    'hydray_v9_part3.bin.zst': 647_094_463,
}
# For other data: set DATA_TAG, list the files in EXPECTED_RECORDS with None as
# the count (no check), e.g. {'hydray_v7_part1.bin.zst': None, ...}.

MIN_MATERIAL = 400      # material imbalance threshold, see the first cell
TOTAL_SB     = 40       # superbatches of 100M samples
STAGE        = 10       # checkpoint to Drive every STAGE superbatches (multiple of 10)
BRANCH       = 'small-net'
NET_ID       = f'hydray-small-{DATA_TAG}-m{MIN_MATERIAL}-{TOTAL_SB}sb'

REPO     = '/content/hydray'
TRAINER  = f'{REPO}/nnue/trainer'
DATA     = '/content/small_data.bin'
CACHE    = f'{DRIVE}/hydray_small_{DATA_TAG}_m{MIN_MATERIAL}.bin.zst'
NET_FILE_BYTES = 795_456

assert TOTAL_SB % STAGE == 0 and STAGE % 10 == 0, 'stage ends must be checkpoints (save_rate 10)'

def find(name):
    hits = glob.glob(f'{DRIVE}/**/{name}', recursive=True)
    assert hits, f'{name} not found on Drive'
    return hits[0]

if os.path.exists(CACHE):
    print(f'cached filtered set found: {CACHE} ({os.path.getsize(CACHE)/2**30:.2f} GiB)')
    PARTS = {}
else:
    PARTS = {name: find(name) for name in EXPECTED_RECORDS}
    for name, path in PARTS.items():
        print(f'{name}: {os.path.getsize(path)/2**30:6.2f} GiB  {path}')
print('net id:', NET_ID)

In [ ]:
# --- Rust, zstd, clone, architecture check ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh('apt-get -qq install -y zstd >/dev/null')
sh(f'rm -rf {REPO} && git clone --depth 1 --branch {BRANCH} https://github.com/ThomasGhione/HydraY {REPO}')
sh(f'cd {REPO} && git log --oneline -1')

# Not ceremonial: a past run trained for an hour an architecture other than the
# one believed, because the clone was wrong and a checkpoint name says nothing
# about its contents. The trainer and the engine header must agree.
tr  = open(f'{TRAINER}/src/bin/trainer_small.rs').read()
hpp = open(f'{REPO}/nnue/network_small.hpp').read()
const = lambda pat, src: int(re.search(pat, src).group(1))
h_tr,  l1_tr  = const(r'const HIDDEN_SIZE: usize = (\d+);', tr),  const(r'const L1_SIZE: usize = (\d+);', tr)
h_cpp, l1_cpp = const(r'int HIDDEN\s*=\s*(\d+);', hpp), const(r'int L1_SIZE\s*=\s*(\d+);', hpp)
assert (h_tr, l1_tr) == (h_cpp, l1_cpp), f'trainer {h_tr}->{l1_tr} vs engine {h_cpp}->{l1_cpp}'
assert (h_tr, l1_tr) == (128, 8), f'not the 128 -> 8 -> 1 net: {h_tr} -> {l1_tr}'
print(f'branch {BRANCH}: {h_tr} -> {l1_tr} -> 1 in trainer and engine')

sh(f'cd {TRAINER} && $HOME/.cargo/bin/cargo build -r --bin datafilter --bin sanity_deep')

In [ ]:
# --- filter (or the cached set) ---
FILTER = f'{TRAINER}/target/release/datafilter'

def filter_parts():
    if os.path.exists(DATA):
        os.remove(DATA)
    kept_total = 0
    for name, path in PARTS.items():
        stats = f'/content/filter_{name}.txt'
        # --long=27: the parts are compressed with a 128 MiB window.
        sh(f'set -o pipefail; zstd -dc --long=27 "{path}" | {FILTER} --min-material {MIN_MATERIAL} '
           f'>> {DATA} 2> {stats}; cat {stats}')
        text = open(stats).read()
        records, kept = map(int, re.search(r'records (\d+), all-zero \d+, kept (\d+)', text).groups())
        if EXPECTED_RECORDS[name] is not None:
            assert records == EXPECTED_RECORDS[name], f'{name}: read {records} records, expected {EXPECTED_RECORDS[name]}'
        kept_total += kept
    assert os.path.getsize(DATA) == 32 * kept_total, 'filtered file size does not match the kept count'
    return kept_total

if os.path.exists(CACHE):
    sh(f'zstd -dc -T0 "{CACHE}" > {DATA}')
    n = int(open(CACHE + '.count').read())
    assert os.path.getsize(DATA) == 32 * n, f'cached set truncated: {os.path.getsize(DATA)} != {32*n}'
else:
    n = filter_parts()
    sh(f'zstd -q -T0 -3 -c {DATA} > "{CACHE}.tmp" && mv "{CACHE}.tmp" "{CACHE}"')
    open(CACHE + '.count', 'w').write(str(n))

print(f'\ntraining set: {n/1e6:.1f}M positions ({os.path.getsize(DATA)/2**30:.2f} GiB), '
      f'{TOTAL_SB} superbatches = {TOTAL_SB*1e8/n:.1f} epochs')

In [ ]:
# --- stage helpers (RUN THIS ALSO WHEN RESTARTING) ---

def stage_cmd(end, start, resume_from):
    """STAGE_END must sit right in front of cargo: `STAGE_END=40 cd dir && cargo ...`
    gives the variable to `cd` only, and the trainer then runs straight to TOTAL_SB
    without stopping. Hence the explicit `env`."""
    args = f'{DATA} {TOTAL_SB} {NET_ID}'
    if resume_from is not None:
        args += f' {start} checkpoints/{NET_ID}-{resume_from}'
    return (f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH '
            f'env CUDA_PATH=/usr/local/cuda STAGE_END={end} '
            f'cargo run -r --bin trainer_small --features cuda -- {args}')

def save_to_drive(end):
    """Copies the checkpoint to Drive and CHECKS it arrived: the Drive mount writes
    through a cache, so an unfinished upload would pass for a finished one."""
    ck  = f'{TRAINER}/checkpoints/{NET_ID}-{end}'
    dst = f'{DRIVE}/{NET_ID}-{end}'
    assert os.path.isdir(ck), f'checkpoint missing locally: {ck}'
    sh(f'rm -rf "{dst}" && cp -r {ck} {DRIVE}/')
    size = lambda p: sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(p) for f in fs)
    assert os.path.isdir(dst) and size(dst) == size(ck), f'incomplete copy on Drive: {dst}'
    print(f'superbatch {end} on Drive ({size(dst)/2**20:.0f} MiB, checked)', flush=True)

def run_stages(done=0):
    assert done % STAGE == 0, f'{done} is not a stage boundary'
    prev = done if done else None
    for k in range(done // STAGE, TOTAL_SB // STAGE):
        end, start = (k + 1) * STAGE, k * STAGE + 1
        print(f'\n===== stage {k+1}/{TOTAL_SB//STAGE}: superbatches {start}-{end} =====', flush=True)
        sh(stage_cmd(end, start, prev))
        save_to_drive(end)
        prev = end

In [ ]:
# --- training ---
# Do NOT run this cell when restarting: use the cell "RESTART" below.
run_stages(done=0)

In [ ]:
# --- final check, copy to Drive ---
final = f'{TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB}/quantised.bin'
size = os.path.getsize(final)
assert size == NET_FILE_BYTES, f'{size} bytes: not the 128 -> 8 -> 1 layout ({NET_FILE_BYTES})'
sh(f'{TRAINER}/target/release/sanity_deep {final}')
sh(f'cp {final} "{DRIVE}/{NET_ID}.nnue" && cmp {final} "{DRIVE}/{NET_ID}.nnue"')
print(f'\nnet on Drive: {DRIVE}/{NET_ID}.nnue')
print('The evals above must be on the big net\'s scale (stm up a queen ~ +1000 or more,')
print('startpos small): the engine mixes the two nets\' evals in one search.')

In [ ]:
# --- RESTART (ONLY if the session died mid-run) ---
#   1. run the cells from "helper" to "stage helpers" (the filter cell reuses the
#      cached set on Drive);
#   2. do NOT run the training cell;
#   3. set RESTART = True and DONE = the last superbatch saved on Drive.
# With RESTART = False this cell does nothing, so "Run all" is safe.
RESTART = False
DONE    = 10

if not RESTART:
    print('restart off (RESTART = False): nothing to do')
else:
    ck = f'{DRIVE}/{NET_ID}-{DONE}'
    assert os.path.isdir(ck), f'checkpoint not found on Drive: {ck}'
    os.makedirs(f'{TRAINER}/checkpoints', exist_ok=True)
    sh(f'cp -r "{ck}" {TRAINER}/checkpoints/')
    run_stages(done=DONE)

## In the engine (`small-net` branch)

    make prod
    ./chess nnue-selftest nnue/net/hydray.nnue 200 <net>.nnue   # incremental == scratch, SIMD == scalar
    ./chess small-eval <net>.nnue < fens.txt                    # small<TAB>big<TAB>fen per line

and in UCI:

    setoption name SmallEvalFile value /path/<net>.nnue

`small-eval` and `sanity_deep <net> - < fens.txt` must print the same small evals
(skip the `info` line): the Rust reader is the reference for the C++ one.